<a href="https://colab.research.google.com/github/Yashika-Bayeen/agentic-ai/blob/main/Day_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
CORPUS = [
    "A Machine Learning Engineer must be fluent in Python and PyTorch or TensorFlow.",
    "MLOps skills include Docker, CI/CD, and model monitoring with tools like MLflow.",
    "Data version control (DVC) and experiment tracking are expected for senior ML roles.",
    "Frontend engineers focus on React, TypeScript, and accessibility (a11y).",
    "Product managers write PRDs and prioritize with frameworks like RICE.",
    "Vector databases (pgvector, Pinecone) power retrieval for RAG systems.",
]
print(f"{len(CORPUS)} documents in corpus.")

6 documents in corpus.


In [ ]:
import math
import re
from collections import Counter

def tokenize(text: str):
    return re.findall(r"[a-z0-9]+", text.lower())

def embed(text: str) -> Counter:
    return Counter(tokenize(text))

def cosine(a: Counter, b: Counter) -> float:
    common = set(a) & set(b)
    dot = sum(a[t] * b[t] for t in common)
    na = math.sqrt(sum(v * v for v in a.values()))
    nb = math.sqrt(sum(v * v for v in b.values()))
    return dot / (na * nb) if na and nb else 0.0

CORPUS_VECS = [embed(doc) for doc in CORPUS]

In [ ]:
def retrieve(query: str, k: int = 3):
    qv = embed(query)
    scored = sorted(
        ((cosine(qv, dv), doc) for dv, doc in zip(CORPUS_VECS, CORPUS)),
        key=lambda x: x[0],
        reverse=True,
    )
    return [doc for score, doc in scored[:k] if score > 0]

query = "What does a machine learning engineer need for MLOps?"
hits = retrieve(query)
for h in hits:
    print("•", h)

• A Machine Learning Engineer must be fluent in Python and PyTorch or TensorFlow.
• Vector databases (pgvector, Pinecone) power retrieval for RAG systems.
• MLOps skills include Docker, CI/CD, and model monitoring with tools like MLflow.


In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.environ["GROQ_API_KEY"])

def answer_grounded(query: str) -> str:
    context = "\n".join(f"- {c}" for c in retrieve(query))
    system = (
        "Answer the question using ONLY the provided context. "
        "If the context is insufficient, say 'I don't have enough information.' "
        "Never invent facts."
    )
    user = f"CONTEXT:\n{context}\n\nQUESTION: {query}"
    r = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user}],
        temperature=0.0,
    )
    return r.choices[0].message.content

print(answer_grounded(query))

A Machine Learning Engineer needs Docker, CI/CD, and model monitoring with tools like MLflow for MLOps.


In [ ]:
print(answer_grounded("What is the capital of France?"))

Paris.
